In [21]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

base = Path("/app")
dataset_dir = base / "ML_Training_datasets" / "Datasets" / "5S"
model_dir = base / "ML_Training_datasets" / "Models" / "5S"
model_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
target_horizons = (10, 20, 30, 40, 50)


In [22]:
csv_files = sorted(dataset_dir.glob("memecoin_training_dataset_*.csv"))
print(f"Found {len(csv_files)} dataset files.")

dfs = []
for file_number, path in enumerate(csv_files, start=1):
    df_part = pd.read_csv(path)
    df_part["source_file"] = path.name

    if "token_id" not in df_part.columns:
        token_stub = path.stem.replace("memecoin_training_dataset_", "token_")
        df_part["token_id"] = token_stub

    if "token_name" not in df_part.columns:
        df_part["token_name"] = df_part["token_id"]

    dfs.append(df_part)

if not dfs:
    raise ValueError(f"No dataset CSVs found in {dataset_dir}")

df = pd.concat(dfs, ignore_index=True)
print(df.shape)
df.head()


Found 19 dataset files.
(231033, 52)


,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,future_return_pct_20,future_close_30,future_return_pct_30,future_close_40,future_return_pct_40,future_close_50,future_return_pct_50,token_id,token_name,source_file
0,76.64,3400727.99,3368076.84,3321586.72,3282056.03,3242820.99,1,38297.54,30072.09,8225.44,...,0.111953,3.479441e+06,1.460234,3.482963e+06,1.562949,3.423032e+06,-0.184629,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
1,74.91,3405330.96,3373597.55,3325683.09,3284907.28,3244644.10,1,38102.21,31678.12,6424.10,...,0.033722,3.478758e+06,1.538611,3.426884e+06,0.024518,3.422471e+06,-0.104287,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
2,77.33,3411744.54,3379979.28,3330189.86,3287990.42,3246593.97,1,38676.55,33077.80,5598.75,...,0.034271,3.473882e+06,0.967165,3.432200e+06,-0.244299,3.421427e+06,-0.557407,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
3,71.02,3414725.35,3384565.92,3334031.00,3290765.64,3248400.39,1,37691.29,34000.50,3690.79,...,-0.096426,3.479687e+06,1.503666,3.425412e+06,-0.079553,3.418651e+06,-0.276783,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
4,71.13,3417259.39,3388765.60,3337742.05,3293496.27,3250194.04,1,36531.59,34506.72,2024.87,...,1.461377,3.473643e+06,1.311900,3.433468e+06,0.140156,3.423067e+06,-0.163203,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv


In [23]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["token_id", "timestamp"]).reset_index(drop=True)

le_supertrend = LabelEncoder()
df["supertrend_trend_enc"] = le_supertrend.fit_transform(df["supertrend_trend"].astype(str))

def add_group_features(g):
    g = g.copy()
    g["rsi_lag_1"] = g["rsi"].shift(1)
    g["rsi_roll_3"] = g["rsi"].rolling(3, min_periods=1).mean()
    g["rsi_roll_5"] = g["rsi"].rolling(5, min_periods=1).mean()
    g["atr_roll_3"] = g["atr"].rolling(3, min_periods=1).mean()
    g["atr_roll_5"] = g["atr"].rolling(5, min_periods=1).mean()
    g["volume_avg_roll_3"] = g["volume_avg"].rolling(3, min_periods=1).mean()
    g["volume_avg_roll_5"] = g["volume_avg"].rolling(5, min_periods=1).mean()
    g["obv_roll_3"] = g["obv"].rolling(3, min_periods=1).mean()
    return g

token_id_backup = df["token_id"].copy()
df = df.groupby("token_id", group_keys=False).apply(add_group_features)
if "token_id" not in df.columns:
    df["token_id"] = token_id_backup.loc[df.index].to_numpy()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)

print(df.shape)
preview_cols = [col for col in ["token_id", "token_name", "timestamp"] if col in df.columns]
df[preview_cols].head()


(231014, 61)


,token_id,token_name,timestamp
0,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 01:48:10+00:00
1,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 01:48:15+00:00
2,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 01:48:40+00:00
3,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 01:48:45+00:00
4,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 01:48:50+00:00


In [24]:
base_feature_cols = [
    "rsi", "ema10", "ema20", "ema50", "ema100", "ema200", "ema_cross",
    "macd_line", "macd_signal", "macd_hist", "vwap", "adx", "plus_di",
    "minus_di", "stoch_k", "stoch_d", "boll_upper", "boll_middle",
    "boll_lower", "boll_percent", "atr", "obv", "supertrend_value",
    "cci", "roc", "momentum3", "volume_sum", "volume_avg", "range_pct",
    "open_rel", "high_rel", "low_rel", "close_rel", "hour", "minute",
    "weekday", "current_close", "supertrend_trend_enc", "rsi_lag_1",
    "rsi_roll_3", "rsi_roll_5", "atr_roll_3", "atr_roll_5",
    "volume_avg_roll_3", "volume_avg_roll_5", "obv_roll_3"
]

optional_feature_cols = [
    "last_5m_buyCount", "last_5m_sellCount", "last_5m_buyVolumeSol",
    "last_5m_sellVolumeSol", "last_5m_priceSol"
]

target_cols = [f"future_return_pct_{h}" for h in target_horizons]
meta_cols = ["token_id", "token_name", "timestamp", "source_file"]

missing_base_feature_cols = [c for c in base_feature_cols if c not in df.columns]
missing_target_cols = [c for c in target_cols if c not in df.columns]
if missing_base_feature_cols:
    raise ValueError(f"Missing required feature columns: {missing_base_feature_cols}")
if missing_target_cols:
    raise ValueError(f"Missing target columns: {missing_target_cols}")

available_optional_feature_cols = [c for c in optional_feature_cols if c in df.columns]
missing_optional_feature_cols = [c for c in optional_feature_cols if c not in df.columns]
feature_cols = base_feature_cols + available_optional_feature_cols

if missing_optional_feature_cols:
    print(f"Optional features not found and skipped: {missing_optional_feature_cols}")

X = df[feature_cols].copy()
tokens = np.array(sorted(df["token_id"].unique()))
print(f"Rows: {len(df)}, features: {len(feature_cols)}, tokens: {len(tokens)}")


Optional features not found and skipped: ['last_5m_buyCount', 'last_5m_sellCount', 'last_5m_buyVolumeSol', 'last_5m_sellVolumeSol', 'last_5m_priceSol']
Rows: 231014, features: 46, tokens: 19


In [25]:
train_tokens, test_tokens = train_test_split(tokens, test_size=0.2, random_state=random_state)
train_tokens, val_tokens = train_test_split(train_tokens, test_size=0.2, random_state=random_state)

train_mask = df["token_id"].isin(train_tokens)
val_mask = df["token_id"].isin(val_tokens)
test_mask = df["token_id"].isin(test_tokens)

split_summary = pd.DataFrame([
    {"split": "train", "tokens": len(train_tokens), "rows": int(train_mask.sum())},
    {"split": "val", "tokens": len(val_tokens), "rows": int(val_mask.sum())},
    {"split": "test", "tokens": len(test_tokens), "rows": int(test_mask.sum())},
])
split_summary


,split,tokens,rows
0,train,12,159895
1,val,3,34947
2,test,4,36172


In [26]:
def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    direction_acc = (np.sign(y_true) == np.sign(y_pred)).mean()
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse,
        "r2": r2_score(y_true, y_pred),
        "direction_acc": direction_acc,
    }

model_params = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
    objective="reg:squarederror",
    tree_method="hist",
)

artifacts = {
    "feature_cols": feature_cols,
    "target_horizons": list(target_horizons),
    "label_encoder_classes": list(le_supertrend.classes_),
    "train_tokens": list(train_tokens),
    "val_tokens": list(val_tokens),
    "test_tokens": list(test_tokens),
}

metrics_rows = []
models = {}

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]
X_test = X.loc[test_mask]

for horizon in target_horizons:
    target_col = f"future_return_pct_{horizon}"
    y_train = df.loc[train_mask, target_col]
    y_val = df.loc[val_mask, target_col]
    y_test = df.loc[test_mask, target_col]

    model = XGBRegressor(**model_params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    val_metrics = regression_metrics(y_val, val_pred)
    test_metrics = regression_metrics(y_test, test_pred)

    metrics_rows.append({
        "horizon": horizon,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })

    models[horizon] = model
    joblib.dump(model, model_dir / f"xgb_return_h{horizon}.joblib")

metrics_df = pd.DataFrame(metrics_rows).sort_values("horizon").reset_index(drop=True)
metrics_df


,horizon,val_mae,val_rmse,val_r2,val_direction_acc,test_mae,test_rmse,test_r2,test_direction_acc
0,10,2.864008,4.504195,-0.471426,0.516210,4.264759,8.257987,-0.716479,0.512164
1,20,3.972518,6.110143,-0.393177,0.517097,6.208294,12.683822,-0.592073,0.523886
2,30,4.980242,7.495133,-0.436449,0.518557,8.466402,18.215768,-0.900860,0.507990
3,40,5.722888,8.393331,-0.380178,0.534266,9.504591,20.182707,-0.629626,0.512966
4,50,6.366936,9.378980,-0.399036,0.522162,10.563555,21.886582,-0.554704,0.499198


In [27]:
joblib.dump(artifacts, model_dir / "training_artifacts.joblib")
metrics_df.to_csv(model_dir / "metrics_summary.csv", index=False)

feature_importance_10 = pd.Series(
    models[10].feature_importances_,
    index=feature_cols,
).sort_values(ascending=False)

feature_importance_10.head(20)


volume_avg_roll_3    0.035759
hour                 0.035205
close_rel            0.029052
ema_cross            0.027710
obv_roll_3           0.027602
ema20                0.027413
obv                  0.027149
vwap                 0.027082
ema100               0.026895
ema200               0.026842
adx                  0.026801
rsi_roll_3           0.026029
volume_avg_roll_5    0.025622
current_close        0.024906
boll_lower           0.024890
volume_sum           0.024734
macd_signal          0.024065
minute               0.023659
weekday              0.023570
low_rel              0.023541
dtype: float32